# [SQL 재현] 소방청 화재발생 통계 — 지역유형 분석

## 단계: 05. 지역유형(군·시·구) 분석 — SQL 재현
- 목표: 시군구 이름의 끝 글자로 지역유형을 나누고, 유형별 화재 현황·발생율 중앙값·장소 구성비를 SQL로 계산해 Python 결과와 대조한다.
- 환경: Jupyter Notebook + sqlite3 (SQL_04에서 만든 analysis2, fire 테이블 사용)
- 대조 기준: nfa_fire_2024.ipynb (3.2, 3.3, 4.3 해석, 5.1, 5.4), nfa_fire_2024.sas (546~549행)
- 참고: 군위군은 경북·대구 두 행으로 나뉘어 있음 (SQL_04 4.5-4). 노트북과 대조하기 위해 두 행을 그대로 둔다.

### 5.0 환경 설정
- 새 노트북이므로 pandas·sqlite3를 불러오고 DB에 연결한다 (노트북을 새로 열거나 커널을 재시작하면 이 셀부터 다시 실행)
- 이번 노트북에서 쓸 테이블(analysis2, fire)이 DB에 있는지 확인

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r'C:\data\sql_practice.db')
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)

,name
0,hira
1,hira_eda
2,hira_model
3,hira_final
4,fire_raw
5,pop
6,fire
7,fire_agg
8,analysis
9,py_result


### 5.1 지역유형 분류
#### 5.1-1 시군구 이름 끝 글자 분포 확인
- Python `str.endswith('군')`, SAS 546~549행 대응
- `SUBSTR(열, -1)`: 글자의 맨 마지막 한 글자를 꺼낸다
- 분류 규칙을 정하기 전에, 끝 글자가 실제로 어떤 것들이 있는지 먼저 확인

In [2]:
q = """
SELECT SUBSTR(SIGUNGU, -1) AS 끝글자,
       COUNT(*)            AS 지역수
FROM analysis2
GROUP BY 끝글자
ORDER BY 지역수 DESC
"""
pd.read_sql(q, conn)

,끝글자,지역수
0,구,104
1,군,83
2,시,66
3,종,1


#### 5.1-2 지역유형 2분류·3분류 열 추가 → fire_region 테이블
- 2분류(REGION_TYPE): 이름이 '군'으로 끝나면 군지역, 나머지는 시구지역 (노트북 3.2)
- 3분류(REGION_CAT): 군 / 구 / 나머지는 시 (노트북 5.4, SAS 546~549행)
- CASE WHEN은 위에서부터 차례로 조건을 보고, 처음 맞는 것 하나만 적용한다

In [3]:
conn.execute("DROP TABLE IF EXISTS fire_region")
conn.execute("""
CREATE TABLE fire_region AS
SELECT *,
        CASE WHEN SIGUNGU LIKE '%군' THEN '군지역'
            ELSE '시구지역' END        AS REGION_TYPE,
        CASE WHEN SUBSTR(SIGUNGU, -1) = '군' THEN '군'
             WHEN SUBSTR(SIGUNGU, -1) = '구' THEN '구'
             ELSE '시' END             AS REGION_CAT
FROM analysis2
""")
conn.commit()

#### 5.1-3 분류 결과 개수 확인
- 2분류와 3분류를 한 표에서 함께 확인 (GROUP BY 두 열)

In [4]:
q = """
SELECT REGION_TYPE, REGION_CAT,
        COUNT(*) AS 지역수
FROM fire_region
GROUP BY REGION_TYPE, REGION_CAT
ORDER BY REGION_TYPE, REGION_CAT
"""
pd.read_sql(q, conn)

,REGION_TYPE,REGION_CAT,지역수
0,군지역,군,83
1,시구지역,구,104
2,시구지역,시,67


### 5.2 지역유형별 집계
#### 5.2-1 지역유형별 화재 현황
- 노트북 5.1 AS-IS 현황 지표 대응 (지역수·화재건수·사망·부상·화재 1,000건당 사망·부상)
- 노트북 3.2 발생율 평균도 함께

In [5]:
q = """
SELECT REGION_TYPE,
       COUNT(*)                                                  AS 지역수,
       SUM(FIRE_CNT)                                             AS 화재건수,
       SUM(DEATH_CNT)                                            AS 사망,
       SUM(INJURY_CNT)                                           AS 부상,
       ROUND(CAST(SUM(DEATH_CNT)  AS REAL) / SUM(FIRE_CNT) * 1000, 2) AS 화재1000건당_사망,
       ROUND(CAST(SUM(INJURY_CNT) AS REAL) / SUM(FIRE_CNT) * 1000, 2) AS 화재1000건당_부상,
       ROUND(AVG(FIRE_RATE_100K), 2)                             AS 발생율_평균
FROM fire_region
GROUP BY REGION_TYPE
"""
pd.read_sql(q, conn)

,REGION_TYPE,지역수,화재건수,사망,부상,화재1000건당_사망,화재1000건당_부상,발생율_평균
0,군지역,83,39462,316,1802,8.01,45.66,1003.36
1,시구지역,171,152047,1258,8585,8.27,56.46,372.99


### 5.3 지역유형별 발생율 중앙값
#### 5.3-1 지역유형별 발생율 중앙값 (PARTITION BY)
- 노트북 3.2 `groupby('REGION_TYPE')['FIRE_RATE_100K'].median()` 대응
- SQL_03 3.4-5(광역시별 중앙값)와 같은 방식: 그룹마다 따로 줄을 세우고 가운데 번호의 값을 꺼낸다

In [6]:
q = """
SELECT REGION_TYPE,
       AVG(x)   AS 발생율_중앙값,
       COUNT(*) AS 사용한_행수
FROM (
    SELECT REGION_TYPE,
           FIRE_RATE_100K AS x,
           ROW_NUMBER() OVER (PARTITION BY REGION_TYPE ORDER BY FIRE_RATE_100K) AS rn,
           COUNT(*)     OVER (PARTITION BY REGION_TYPE)                         AS cnt
    FROM fire_region
)
WHERE rn IN ((cnt + 1) / 2, (cnt + 2) / 2)
GROUP BY REGION_TYPE
"""
pd.read_sql(q, conn)

,REGION_TYPE,발생율_중앙값,사용한_행수
0,군지역,1017.06,1
1,시구지역,320.37,1


#### 5.3-2 (선택) 군위군을 뺐을 때 지역유형별 발생율 평균
- 군위군 두 행(경북·대구)은 각각 5년 중 일부 기간만 담고 있음 (SQL_04 4.5-4)
- 두 행을 뺐을 때 군지역 평균이 얼마나 달라지는지 확인 (민감도 확인)

In [7]:
q = """
SELECT REGION_TYPE,
       COUNT(*)                                                       AS 지역수,
       ROUND(AVG(FIRE_RATE_100K), 2)                                  AS 발생율_평균,
       COUNT(CASE WHEN SIGUNGU <> '군위군' THEN 1 END)                AS 군위군제외_지역수,
       ROUND(AVG(CASE WHEN SIGUNGU <> '군위군' THEN FIRE_RATE_100K END), 2) AS 군위군제외_평균
FROM fire_region
GROUP BY REGION_TYPE
"""
pd.read_sql(q, conn)

,REGION_TYPE,지역수,발생율_평균,군위군제외_지역수,군위군제외_평균
0,군지역,83,1003.36,81,1009.32
1,시구지역,171,372.99,171,372.99


### 5.4 시도별 발생율 중앙값
#### 5.4-1 시도별 발생율 중앙값 (PARTITION BY SIDO)
- 노트북 3.3 `groupby('SIDO')['FIRE_RATE_100K'].median()` 대응
- 5.3-1과 같은 틀에서 그룹 열만 SIDO로 바꾼다

In [8]:
q = """
SELECT REGION_TYPE,
       COUNT(*)                                                             AS 지역수,
       ROUND(AVG(FIRE_RATE_100K), 2)                                        AS 발생율_평균,
       COUNT(CASE WHEN SIGUNGU <> '군위군' THEN 1 END)                      AS 군위군제외_지역수,
       ROUND(AVG(CASE WHEN SIGUNGU <> '군위군' THEN FIRE_RATE_100K END), 2) AS 군위군제외_평균
FROM fire_region
GROUP BY REGION_TYPE
"""
pd.read_sql(q, conn)

,REGION_TYPE,지역수,발생율_평균,군위군제외_지역수,군위군제외_평균
0,군지역,83,1003.36,81,1009.32
1,시구지역,171,372.99,171,372.99


### 5.4 시도별 발생율 중앙값
#### 5.4-1 시도별 발생율 중앙값 (PARTITION BY SIDO)
- 노트북 3.3 `groupby('SIDO')['FIRE_RATE_100K'].median()` 대응
- 5.3-1과 같은 틀에서 그룹 열만 SIDO로 바꾼다

In [11]:
q = """
SELECT SIDO,
       AVG(x)   AS 발생율_중앙값,
       COUNT(*) AS 사용한_행수
FROM (
    SELECT SIDO,
           FIRE_RATE_100K AS x,
           ROW_NUMBER() OVER (PARTITION BY SIDO ORDER BY FIRE_RATE_100K) AS rn,
           COUNT(*)     OVER (PARTITION BY SIDO)                         AS cnt
    FROM fire_region
)
WHERE rn IN ((cnt + 1) / 2, (cnt + 2) / 2)
GROUP BY SIDO
ORDER BY 발생율_중앙값 DESC
"""
pd.read_sql(q, conn)

,SIDO,발생율_중앙값,사용한_행수
0,전라남도,1059.155,2
1,전북특별자치도,990.640,1
2,강원특별자치도,891.080,2
3,경상북도,818.090,2
4,경상남도,800.295,2
5,충청남도,669.100,2
6,충청북도,544.170,2
7,제주특별자치도,492.565,2
8,부산광역시,365.140,2
9,대전광역시,316.360,1
